## 380_RAG_demo

### This is intended as a scaffold code for HW\#4 (**LLM-RAG**).

In [1]:
# Setup  (do not modify)
# ----------------------------
# Colab already has torch, transformers, and sentence-transformers pre-installed.
# Only install the LangChain ecosystem packages that are not pre-installed.
# Do NOT use -U or --force-reinstall here — that breaks Colab's torch/transformers.
!pip install -q \
    langchain langchain-core langchain-community \
    langchain-huggingface langchain-openai langchain-text-splitters \
    faiss-cpu openai tiktoken pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [11]:
# Imports  (do not modify)
# ----------------------------
import os, re, json as _json, time
import requests
from typing import List, Dict, Tuple
from IPython.display import Markdown, display

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [3]:
# Load API keys from CoLab secret keys  (do not modify)
from google.colab import userdata

langchain_api = userdata.get('langchain_api')
openai_api    = userdata.get('openai_api')
hf_token      = userdata.get('HF_TOKEN')

os.environ['OPENAI_API_KEY']           = openai_api
os.environ['LANGCHAIN_API_KEY']        = langchain_api
os.environ['HUGGINGFACEHUB_API_TOKEN'] = hf_token

## Part 1: System and Chain Building

In [13]:
# 1. Load and process a document from GitHub
url = "https://raw.githubusercontent.com/ntomuro/CSC380/main/HW4-LLM_RAG/data/Human-Nutrition-2020-Edition-1598491699.txt"
response = requests.get(url)
response.raise_for_status()
documents = [Document(page_content=response.text, metadata={"source": url})]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
docs = [
    Document(
        page_content=chunk_text.page_content, # Access the page_content attribute
        metadata={
            "doc_id": "doc1",
            "chunk_id": i,
            "source": "nutrition_book"
        }
    )
    for i, chunk_text in enumerate(chunks)
]
print(f"Created {len(docs)} chunks")

# 2. Create embeddings and store in FAISS (in-memory, no disk writes)
embedding_model = "all-MiniLM-L6-v2"
embedding_function = HuggingFaceEmbeddings(model_name=embedding_model)
vector_store = FAISS.from_documents(docs, embedding_function)

# 3. Set up retriever and LLM
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 4. Define RAG prompt
prompt = PromptTemplate.from_template(
    "Use the following context to answer the question.\n"
    "If you don't know the answer, say so — don't make one up.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

# 5. Helper: run one query through the LCEL pipeline
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def run_query(query):
    retrieved_docs = retriever.invoke(query)
    chain = (
        {"context": lambda _: format_docs(retrieved_docs),
         "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain.invoke(query), retrieved_docs


Created 4031 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Part 2: System and Chain Execution

In [18]:
# 6. Query the RAG system — store results for reuse in the judge step

# 3 answerable questions and 3 unanswerable questions
queries = [
    "Do you die from diebetes?  If so, what's the chance of surviving?",
    "Is diabetes hereditary? My father was diabetic. What's my chance of getting" +
    " it? What should I be watching (food, exercise etc.) to prevent not becoming diabetic?",
    "Is there an exact number of grams of carbs you can eat each day that will" +
    " always reverse prediabetes within three months? If yes, what proof shows this works for everyone?",
    "How much is diabetes covered in typical insurance? What expenses or prescriptions are covered?",
    "What is the future vision for healthcare in the US?",
    "How do different diabetes medications compare in preventing the progression" +
    " from prediabetes to type 2 diabetes, and what are the precise percentages of effectiveness reported for each treatment?",
]

rag_results = []
for idx, query in enumerate(queries):
    answer, retrieved_docs = run_query(query)
    doc_ids = [d.metadata.get("chunk_id") for d in retrieved_docs]
    rag_results.append({"query": query, "answer": answer, "retrieved_docs": retrieved_docs})
    print(f"\n-------------------\n[Query {idx}] {query}")
    print(f"[Answer {idx}, retrieved doc_ids: {doc_ids}]")
    print(f"\n{answer}")


-------------------
[Query 0] Do you die from diebetes?  If so, what's the chance of surviving?
[Answer 0, retrieved docids: [3848, 3131, 3828]]

Yes, diabetes can lead to death, particularly if it is not managed properly. People with diabetes are at a higher risk of dying from complications such as cardiovascular disease, kidney failure, and other serious health issues. The context mentions that Type 2 diabetes causes about seventy thousand deaths annually in the United States. 

As for the chance of surviving, it depends on various factors, including the type of diabetes, how well it is managed, access to medical care, and individual health conditions. With proper management, including medication, lifestyle changes, and regular monitoring, many individuals with diabetes can live long and healthy lives. However, untreated diabetes or severe complications can significantly reduce survival chances.

-------------------
[Query 1] Is diabetes hereditary? My father was diabetic. What's my

## Part 3: LLM as Judge

In [19]:
# 7. LLM-as-Judge: evaluate stored results — no re-retrieval, no re-generation
eval_prompt = PromptTemplate.from_template(
    "You are evaluating a Retrieval-Augmented Generation (RAG) system.\n\n"
    "Query: {query}\n"
    "Retrieved context (excerpt): {context}\n"
    "Answer: {answer}\n\n"
    "Score each item on a 1-3 scale:\n"
    "  - retrieval_score: 1=irrelevant, 2=partially relevant, 3=highly relevant\n"
    "  - answer_score:    1=incorrect,  2=partially correct,  3=correct & complete\n\n"
    "Respond with ONLY valid JSON, nothing else:\n"
    '{{"retrieval_score": <int>, "answer_score": <int>, "reason": "<one sentence>"}}'
)
judge_llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_chain = eval_prompt | judge_llm | StrOutputParser()

for r in rag_results:
    context_excerpt = r["retrieved_docs"][0].page_content[:500] if r["retrieved_docs"] else "(none)"
    raw = judge_chain.invoke({"query": r["query"], "context": context_excerpt, "answer": r["answer"]})
    try:
        scores = _json.loads(raw)
        print(f"\nQuery:  {r['query']}")
        print(f"[Judge] retrieval={scores['retrieval_score']}, "
              f"answer={scores['answer_score']} — {scores['reason']}")
    except (_json.JSONDecodeError, KeyError):
        print(f"[Judge] Could not parse response: {raw}")


Query:  Do you die from diebetes?  If so, what's the chance of surviving?
[Judge] retrieval=3, answer=3 — The retrieved context provides relevant information about the risks associated with diabetes and the potential for death, and the answer accurately reflects this information while addressing the query comprehensively.

Query:  Is diabetes hereditary? My father was diabetic. What's my chance of getting it? What should I be watching (food, exercise etc.) to prevent not becoming diabetic?
[Judge] retrieval=3, answer=3 — The retrieved context provides relevant information about the hereditary nature of diabetes and includes practical advice on prevention, which aligns well with the query.

Query:  Is there an exact number of grams of carbs you can eat each day that will always reverse prediabetes within three months? If yes, what proof shows this works for everyone?
[Judge] retrieval=2, answer=3 — The retrieved context provides relevant information about low-carb diets but does not di